<a href="https://colab.research.google.com/github/Alenushka2013/ML_for_people_tasks/blob/main/HW_6_Using_prompts_and_agents_in_Langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


### Завдання 1: Виклик LLM з базовим промптом

Створіть можливість викликати LLM зі звичайним текстовим промптом.

Промпт має дозвляти отримати інформацію простою мовою на певну тему. В цьому завданні ми хочемо дізнатись про тему "Квантові обчислення".

Відповідь моделі повинна містити визначення, ключові переваги та поточні дослідження в цій галузі.

Обмежте відповідь до 200 символів і пропишіть в промпті аби відповідь була короткою (це зекономить Вам час і гроші на згенеровані токени).

В якості LLM можна скористатись як моделлю з HugginFace (рекомендую Mistral), так і ChatGPT4 або ChatGPT3. В обох випадках треба імпортувати потрібну "обгортку" (тобто клас, який дозволить ініціювати модель) з LangChain для виклику LLM за API, а також зчитати особистий токен з файла, наприклад, `creds.json`, який розміщений у Вас локально і Ви НЕ здаєте його в ДЗ і НЕ комітите в git 😏

Встановіть своє значення температури на свій розсуд (тут немає правильного чи неправильного значення) і напишіть, чому ви обрали саме таке значення для цього завдання.  

Запити можна робити як українською, так і англійською - орієнтуйтесь на те, де і чи хочете ви потім лишити цей проєкт і відповідна яка мова буде пасувати більше. В розвʼязках промпти - українською.

In [ ]:
!pip -q install langchain langchain_openai huggingface_hub openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 1.8 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import InferenceClient
import json

with open("creds.json") as file:
    creds = json.load(file)

client = InferenceClient(
    "mistralai/Mistral-7B-Instruct-v0.2",
    token=creds["HUGGINGFACEHUB_API_TOKEN"]
)

prompt = """
Тема: "Квантові обчислення".
Дай визначення, ключові переваги та поточні дослідження.
Відповідь простою мовою, обмеж до 200 символів, будь лаконічним.
"""

response = client.chat_completion(
    messages=[{"role": "user", "content": prompt}],
    max_tokens=200,
    temperature=0
)

print(response.choices[0].message["content"])


 Квантове обчислення - це область комп'ютерних наук, яка використовує квантову механіку для розв'язання складних математичних задач швидше, ніж класичні комп'ютери. Ключові переваги: паралельність обчислень (квабіт може бути у двох станах одночасно), висока швидкість для специфічних задач (наприклад, шифрування та оптимізація). Поточні дослідження: розвиток квантових алгоритмів, створення надійніших квантових бітів, покращення квантових комп'ютерів.


Температура рівна 0 відповідає мінімальній креативності мовної моделі. Визначення наукового терміну хотілося б отримати точним, без зайвих вигадок.

### Завдання 2: Створення параметризованого промпта для генерації тексту
Тепер ми хочемо оновити попередній фукнціонал так, аби в промпт ми могли передавати тему як параметр. Для цього скористайтесь `PromptTemplate` з `langchain` і реалізуйте параметризований промпт та виклик моделі з ним.

Запустіть оновлений функціонал (промпт + модел) для пояснень про теми
- "Баєсівські методи в машинному навчанні"
- "Трансформери в машинному навчанні"
- "Explainable AI"

Виведіть результати відпрацювання моделі на екран.

In [ ]:
from langchain.prompts import PromptTemplate

# --- Параметризований промпт через LangChain ---
template = """
Тема: "{topic}".
Дай визначення, ключові переваги та поточні дослідження.
Відповідь простою мовою, обмеж до 200 символів, будь лаконічним.
"""
prompt = PromptTemplate(template=template, input_variables=["topic"])

# --- Список тем ---
topics = [
    "Баєсівські методи в машинному навчанні",
    "Трансформери в машинному навчанні",
    "Explainable AI"
]

# --- Генерація відповідей ---
for t in topics:
    formatted_prompt = prompt.format(topic=t)

    response = client.chat_completion(
        messages=[{"role": "user", "content": formatted_prompt}],
        max_tokens=200,
        temperature=0.1
    )

    print(f"Тема: {t}")
    print(response.choices[0].message["content"])
    print("-" * 50)

Тема: Баєсівські методи в машинному навчанні
 Баєсівські методи - форма машинного навчання, де використовується теорія імовірностей для визначення ймовірності класу на основі прийнятих припущень. Вони дозволяють оновлювати знання за новою інформацією, маючи можливість враховувати пріорні знання. Ключові переваги: гнучкість, можливість враховувати непевність, ефективність при малих об'ємах даних. Поточні дослідження: дослідження ефективних алгоритмів, розвиток методів для великих даних, поєднання з глибинними нейронними мережами.
--------------------------------------------------
Тема: Трансформери в машинному навчанні
 Transformers in Machine Learning:

Transformers are a type of neural network model introduced by Google in 2017, revolutionizing natural language processing (NLP) with their ability to handle long-range dependencies. In ML, they're used for tasks like text generation, translation, and summarization.

Key benefits:
1. Long-range dependency modeling: Transformers can proce



### Завдання 3: Використання агента для автоматизації процесів
Створіть агента, який допоможе автоматично шукати інформацію про останні наукові публікації в різних галузях. Наприклад, агент має знайти 5 останніх публікацій на тему штучного інтелекту.

**Кроки:**
1. Налаштуйте агента типу ReAct в LangChain для виконання автоматичних запитів.
2. Створіть промпт, який спрямовує агента шукати інформацію в інтернеті або в базах даних наукових публікацій.
3. Агент повинен видати список публікацій, кожна з яких містить назву, авторів і короткий опис.

Для взаємодії з пошуком там необхідно створити `Tool`. В лекції ми використовували `serpapi`. Можна продовжити користуватись ним, або обрати інше АРІ для пошуку (вони в тому числі є безкоштовні). Перелік різних АРІ, доступних в langchain, і орієнтир по вартості запитів можна знайти в окремому документі [тут](https://hannapylieva.notion.site/API-12994835849480a69b2adf2b8441cbb3?pvs=4).

Лишаю також нижче приклад використання одного з безкоштовних пошукових АРІ - DuckDuckGo (не потребує створення токена!)  - можливо він вам сподобається :)


In [ ]:
!pip install -q langchain_community duckduckgo_search

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

search.invoke("Obama's first name?")

"2 of 2. Barack Obama: timeline Key events in the life of Barack Obama. Barack Obama (born August 4, 1961, Honolulu, Hawaii, U.S.) is the 44th president of the United States (2009-17) and the first African American to hold the office. Before winning the presidency, Obama represented Illinois in the U.S. Senate (2005-08). Since the office was established in 1789, 45 men have served in 46 presidencies. The first president, George Washington, won a unanimous vote of the Electoral College. [4] Grover Cleveland served two non-consecutive terms and is therefore counted as the 22nd and 24th president of the United States, giving rise to the discrepancy between the ... Here is a list of the presidents and vice presidents of the United States along with their parties and dates in office. ... Chester A Arthur: Twenty-First President of the United States. 10 Interesting Facts About James Buchanan. Martin Van Buren - Eighth President of the United States. Quotes From Harry S. Truman. Table of Cont

In [ ]:
!pip install smolagents -q

In [ ]:
!pip install langchain-huggingface

In [ ]:
!pip install -U ddgs

In [ ]:
from huggingface_hub import InferenceClient

hf_token = creds["HUGGINGFACEHUB_API_TOKEN"]

# створюємо клієнт
client = InferenceClient(
    model="mistralai/Mistral-7B-Instruct-v0.2",
    token=hf_token
)

# запит
prompt = "Напиши коротке резюме про переваги використання штучного інтелекту в медицині."

response = client.chat_completion(
    messages=[{"role": "user", "content": prompt}],
    max_tokens=200
)

print("\n📌 Відповідь моделі:")
print(response.choices[0].message["content"])



📌 Відповідь моделі:
 Artificial Intelligence (AI) is revolutionizing various industries, and medicine is no exception. The application of AI in healthcare brings numerous benefits, enhancing the accuracy of diagnoses, improving patient care, and streamlining administrative tasks. Here are some advantages of using AI in medicine:

1. Diagnosis and Early Disease Detection: AI algorithms can analyze large datasets, including electronic health records, medical imaging, and patient symptoms, enabling earlier and more accurate diagnoses. This can lead to better treatment outcomes and reduced healthcare costs.

2. Personalized Medicine: AI analysis of a patient's genetic data, treatment history, and lifestyle can lead to personalized treatment plans, improving efficacy and reducing potential side effects.

3. Predictive Analytics: AI can analyze trends in patient data, predicting the likelihood of diseases or health complications. This enables preventive measures and early intervention, impr

In [ ]:
!pip install -U langchain-openai

In [ ]:
!pip install langchain duckduckgo-search requests transformers

In [ ]:
from langchain.tools import Tool
from langchain.agents import initialize_agent, AgentType
from langchain_community.llms import HuggingFacePipeline
from duckduckgo_search import DDGS
from transformers import pipeline
import requests

# 🔍 DuckDuckGo Tool
def duckduckgo_search(query: str) -> str:
    results = []
    with DDGS() as ddgs:
        for r in ddgs.text(query, max_results=5):
            results.append(f"- {r['title']}\n{r['body']}\n{r['href']}")
    return "\n\n".join(results)

search_tool = Tool.from_function(
    func=duckduckgo_search,
    name="DuckDuckGoSearch",
    description="Use this tool to search for recent scientific publications on any topic."
)

# 📚 arXiv Tool
def arxiv_search(query: str) -> str:
    url = f"http://export.arxiv.org/api/query?search_query=all:{query}&start=0&max_results=5"
    response = requests.get(url)
    return response.text[:2000]  # обрізаємо для простоти

arxiv_tool = Tool.from_function(
    func=arxiv_search,
    name="ArxivSearch",
    description="Use this tool to find academic papers from arXiv."
)

# 🧠 Локальна модель (GPT-2 або інша)
generator = pipeline("text-generation", model="gpt2", max_new_tokens=300)
llm = HuggingFacePipeline(pipeline=generator)

# 🧠 Агент ReAct
agent = initialize_agent(
    tools=[search_tool, arxiv_tool],
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True  # дозволяє обробку нестандартних відповідей
)

# 📌 Запит
task = """
Find 5 recent scientific publications on the topic of "artificial intelligence".
For each one, provide:
- Title
- Authors
- A brief summary (1–2 sentences)
"""

# 🚀 Виконання
result = agent.run(task)

print("\n📚 Результати:")
print(result)

Device set to use cpu
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.




> Entering new AgentExecutor chain...


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Parsing LLM output produced both a final answer and a parse-able action:: Answer the following questions as best you can. You have access to the following tools:

DuckDuckGoSearch(query: str) -> str - Use this tool to search for recent scientific publications on any topic.
ArxivSearch(query: str) -> str - Use this tool to find academic papers from arXiv.

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [DuckDuckGoSearch, ArxivSearch]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: 
Find 5 recent scientific publications on the topic of "artificial intelligence".
For each one, provide:
- Title
- Authors
- A brief summary (1–2 sentences)

Thought: What are you

This is a friendly reminder - the current text generation call will exceed the model's predefined maximum length (1024). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.


IndexError: index out of range in self

In [ ]:
#!pip install feedparser

In [ ]:
import feedparser
from urllib.parse import quote

def arxiv_search(topic, max_results=5):
    """
    Шукає останні публікації на arXiv за темою topic.
    Повертає список словників з назвою, авторами та описом.
    """
    base_url = "http://export.arxiv.org/api/query?"
    encoded_topic = quote(topic)  # <-- кодуємо тему у URL
    search_query = f"search_query=all:{encoded_topic}&start=0&max_results={max_results}&sortBy=submittedDate&sortOrder=descending"
    url = base_url + search_query
    feed = feedparser.parse(url)

    results = []
    for entry in feed.entries:
        title = entry.title.replace("\n", " ").strip()
        authors = ", ".join([author.name for author in entry.authors])
        summary = entry.summary.replace("\n", " ").strip()
        results.append({
            "Topic": topic,
            "Title": title,
            "Authors": authors,
            "Summary": summary
        })
    return results


In [ ]:
topics = [
    "Artificial Intelligence",
    "machine learning algorithms",
    "Explainable AI"
]

all_results = []
for t in topics:
    all_results.extend(arxiv_search(t))

# Вивід результатів
for topic in topics:
    print(f"=== Тема: {topic} ===")
    for r in [x for x in all_results if x["Topic"] == topic]:
        print(f"Назва: {r['Title']}")
        print(f"Автори: {r['Authors']}")
        print(f"Опис: {r['Summary']}\n")
    print("-" * 80)

=== Тема: Artificial Intelligence ===
Назва: The Demon is in Ambiguity: Revisiting Situation Recognition with Single   Positive Multi-Label Learning
Автори: Yiming Lin, Yuchen Niu, Shang Wang, Kaizhu Huang, Qiufeng Wang, Xiao-Bo Jin
Опис: Context recognition (SR) is a fundamental task in computer vision that aims to extract structured semantic summaries from images by identifying key events and their associated entities. Specifically, given an input image, the model must first classify the main visual events (verb classification), then identify the participating entities and their semantic roles (semantic role labeling), and finally localize these entities in the image (semantic role localization). Existing methods treat verb classification as a single-label problem, but we show through a comprehensive analysis that this formulation fails to address the inherent ambiguity in visual event recognition, as multiple verb categories may reasonably describe the same image. This paper makes t



### Завдання 4: Створення агента-помічника для вирішення бізнес-задач

Створіть агента, який допомагає вирішувати задачі бізнес-аналітики. Агент має допомогти користувачу створити прогноз по продажам на наступний рік враховуючи рівень інфляції і погодні умови. Агент має вміти використовувати Python і ходити в інтернет аби отримати актуальні дані.

**Кроки:**
1. Налаштуйте агента, який працюватиме з аналітичними даними, заданими текстом. Користувач пише

```
Ми експортуємо апельсини з Бразилії. В 2021 експортували 200т, в 2022 - 190т, в 2023 - 210т, в 2024 який ще не закінчився - 220т. Зроби оцінку скільки ми зможемо експортувати апельсинів в 2025 враховуючи погодні умови в Бразилії і попит на апельсини в світі виходячи з економічної ситуації.
```

2. Створіть запит до агента, що містить чітке завдання – видати результат бізнес аналізу або написати, що він не може цього зробити і запит користувача (просто може бути все одним повідомлленням).

3. Запустіть агента і проаналізуйте результати. Що можна покращити?
